# Assignment 2: Hand gesture recognition, level-up with SAM2 + DINOv2

**Idea.** In A1 MobileNetV2 reached ~99% on train but only ~60% on test, because the test videos had a different background and clothing (grey curtain, grey sleeve).
Here we attack exactly that problem:

1. **SAM2** cuts the hand out of every training frame.
2. The hand is pasted on many **synthetic backgrounds** (+ fake grey sleeve) -> synthetic training data (always labelled as synthetic).
3. **DINOv2** (SOTA self-supervised features) replaces the A1 backbone.
4. 2x2 ablation: {MobileNetV2, DINOv2} x {original data, original + synthetic}, 3 seeds each.
5. Explainability (attention rollout), stress-test per background, latency, Gradio app.

**Run order:** Runtime -> Change runtime type -> **T4 GPU**, then run all cells top to bottom.

## 0. Setup

In [ ]:
!pip -q install transformers gradio opencv-python-headless scikit-learn seaborn
!pip -q install "git+https://github.com/facebookresearch/sam2.git"

In [ ]:
import os, zipfile, time, json, random, copy
from pathlib import Path
import numpy as np, cv2, torch, torch.nn as nn
import matplotlib.pyplot as plt, pandas as pd, seaborn as sns
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import torchvision, torchvision.transforms as T

dev = "cuda" if torch.cuda.is_available() else "cpu"
DEV_NAME = torch.cuda.get_device_name(0) if dev == "cuda" else "CPU"
CLASSES = ["peace", "stop", "thumbs_up"]
SEEDS = [0, 1, 2]
N_SYNTH_PER_IMG = 4            # synthetic variants per training image
DINO_NAME = "facebook/dinov2-base"
OUT = Path("/content/outputs"); OUT.mkdir(exist_ok=True)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print("device:", dev, DEV_NAME)

## 1. Load the A1 dataset
Put your dataset zip on Google Drive and set `ZIP_PATH`. If it is not found, you will be asked to upload it.
Expected: folders containing the split (`train` / `val` / `test`) and the class (`peace`, `stop`, `thumbs_up`) anywhere in the path.

In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
ZIP_PATH = "/content/drive/MyDrive/dataset.zip"     # <-- CHANGE to your file
DATA_DIR = Path("/content/data"); DATA_DIR.mkdir(exist_ok=True)
if not Path(ZIP_PATH).exists():
    print("Zip not found on Drive, please upload it:")
    up = files.upload(); ZIP_PATH = list(up.keys())[0]
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_DIR)

EXT = {".jpg", ".jpeg", ".png"}
SPLIT_ALIAS = {"train": "train", "training": "train", "val": "val", "valid": "val", "validation": "val", "test": "test"}
norm = lambda s: s.lower().replace(" ", "_").replace("-", "_")

data = {s: ([], []) for s in ("train", "val", "test")}
skipped = []
for p in sorted(DATA_DIR.rglob("*")):
    if p.suffix.lower() in EXT and "__macosx" not in str(p).lower():
        parts = [norm(x) for x in p.relative_to(DATA_DIR).parts]
        split = next((SPLIT_ALIAS[x] for x in parts if x in SPLIT_ALIAS), None)
        cls = next((c for c in CLASSES if c in parts), None)
        if split is None or cls is None:
            skipped.append(str(p)); continue
        im = Image.open(p).convert("RGB").resize((224, 224))
        data[split][0].append(np.array(im)); data[split][1].append(CLASSES.index(cls))

if len(data["train"][0]) == 0:
    print("Could not detect splits/classes. Example paths:")
    print("\n".join(str(p.relative_to(DATA_DIR)) for p in list(DATA_DIR.rglob("*"))[:20]))
    raise SystemExit("Fix folder names or send me this printout.")

Xtr, ytr = data["train"][0], np.array(data["train"][1])
Xva, yva = data["val"][0], np.array(data["val"][1])
Xte, yte = data["test"][0], np.array(data["test"][1])
print("train/val/test:", len(Xtr), len(Xva), len(Xte), "| skipped files:", len(skipped))
for n, y in [("train", ytr), ("val", yva), ("test", yte)]:
    print(n, {CLASSES[i]: int((y == i).sum()) for i in range(3)})

## 2. SAM2: cut the hand out of every frame

In [ ]:
from sam2.sam2_image_predictor import SAM2ImagePredictor
predictor = SAM2ImagePredictor.from_pretrained("facebook/sam2.1-hiera-small")
use_bf16 = dev == "cuda" and torch.cuda.get_device_capability()[0] >= 8

In [ ]:
def hand_mask(img):
    # prompt = one positive point in the image centre (frames were centre-cropped around the hand)
    with torch.inference_mode(), torch.autocast(dev, dtype=torch.bfloat16, enabled=use_bf16):
        predictor.set_image(img)
        masks, scores, _ = predictor.predict(point_coords=np.array([[112, 112]]),
                                             point_labels=np.array([1]), multimask_output=True)
    for m, s in sorted(zip(masks, scores), key=lambda t: -t[1]):
        if 0.04 < m.mean() < 0.70:               # reject empty / whole-frame masks
            return m.astype(np.uint8)
    return None

t0 = time.time()
train_masks = [hand_mask(x) for x in Xtr]
test_masks = [hand_mask(x) for x in Xte]
print(f"SAM2 done in {time.time()-t0:.0f}s | valid masks: train {sum(m is not None for m in train_masks)}/{len(Xtr)}, "
      f"test {sum(m is not None for m in test_masks)}/{len(Xte)}")

In [ ]:
# visual check of the masks (put this figure into the report)
idx = np.random.default_rng(1).choice(len(Xtr), 8, replace=False)
fig, ax = plt.subplots(2, 8, figsize=(20, 5))
for j, i in enumerate(idx):
    ax[0, j].imshow(Xtr[i]); ax[0, j].set_title(CLASSES[ytr[i]]); ax[0, j].axis("off")
    cut = Xtr[i].copy()
    if train_masks[i] is not None: cut[train_masks[i] == 0] = 255
    ax[1, j].imshow(cut); ax[1, j].set_title("SAM2 cut-out"); ax[1, j].axis("off")
plt.tight_layout(); plt.savefig(OUT / "fig_sam2_masks.png", dpi=120); plt.show()

## 3. Synthetic data: new backgrounds + fake grey sleeve
Procedural backgrounds (no external data, no real people): `solid, gradient, curtain, wood, noise`.
The family `photo` (natural images from scikit-image) is **held out**: never used for training, only in the stress-test.

In [ ]:
H = W = 224
def bg_solid(r):
    return np.full((H, W, 3), r.integers(20, 255, 3), np.uint8)
def bg_gradient(r):
    c1, c2 = r.integers(0, 255, 3), r.integers(0, 255, 3)
    t = np.linspace(0, 1, H if r.random() < .5 else W)[:, None, None]
    g = (c1 * (1 - t) + c2 * t)
    g = np.broadcast_to(g, (H, W, 3)) if g.shape[0] == H else np.broadcast_to(g.transpose(1, 0, 2), (H, W, 3))
    return g.astype(np.uint8)
def bg_curtain(r):                                  # vertical folds, greyish (like the val/test curtain)
    base = r.integers(90, 190); x = np.arange(W)
    folds = np.sin(x * r.uniform(.08, .25) + r.uniform(0, 6)) * r.uniform(10, 30)
    img = base + folds[None, :, None] + r.normal(0, 4, (H, W, 1))
    return np.clip(np.broadcast_to(img, (H, W, 3)) + r.integers(-8, 8, 3), 0, 255).astype(np.uint8)
def bg_wood(r):                                     # brown rows with grain
    col = np.array([r.integers(140, 200), r.integers(95, 145), r.integers(50, 95)])
    y = np.arange(H)[:, None]
    grain = np.sin(y * r.uniform(.15, .5) + cv2.resize(r.normal(0, 3, (H // 8, 1)), (1, H))) * 14
    img = col[None, None, :] + grain[..., None] + r.normal(0, 5, (H, W, 1))
    return np.clip(np.broadcast_to(img, (H, W, 3)), 0, 255).astype(np.uint8)
def bg_noise(r):
    small = r.integers(0, 255, (8, 8, 3)).astype(np.uint8)
    return cv2.resize(small, (W, H), interpolation=cv2.INTER_CUBIC)

PHOTOS = []
try:
    from skimage import data as skd
    for n in ["astronaut", "coffee", "chelsea", "rocket", "camera", "brick", "grass", "gravel", "moon", "page"]:
        try:
            im = getattr(skd, n)()
            if im.ndim == 2: im = np.stack([im] * 3, -1)
            PHOTOS.append(im[..., :3].astype(np.uint8))
        except Exception:
            pass
except Exception:
    pass
def bg_photo(r):
    if not PHOTOS: return bg_noise(r)
    im = PHOTOS[r.integers(len(PHOTOS))]
    s = int(min(im.shape[:2]) * r.uniform(.4, 1.0))
    y0 = r.integers(0, im.shape[0] - s + 1); x0 = r.integers(0, im.shape[1] - s + 1)
    return cv2.resize(im[y0:y0 + s, x0:x0 + s], (W, H))

BG = {"solid": bg_solid, "gradient": bg_gradient, "curtain": bg_curtain, "wood": bg_wood, "noise": bg_noise, "photo": bg_photo}
TRAIN_FAMS = ["solid", "gradient", "curtain", "wood", "noise"]     # "photo" is held out

def add_sleeve(bg, r):
    out = bg.copy(); d = int(r.integers(50, 110)); g = int(r.integers(50, 140)); col = np.array([g, g, g + int(r.integers(0, 12))])
    pts = [[(0, 0), (d, 0), (int(d * .6), H), (0, H)], [(W, 0), (W - d, 0), (W - int(d * .6), H), (W, H)],
           [(0, H), (0, H - d), (W, H - int(d * .6)), (W, H)], [(0, 0), (W, 0), (W, d), (0, int(d * .6))]][int(r.integers(4))]
    poly = np.zeros((H, W), np.uint8); cv2.fillConvexPoly(poly, np.array(pts, np.int32), 1)
    tex = r.normal(0, 6, (H, W, 1))
    return np.where(poly[..., None] == 1, np.clip(col + tex, 0, 255), out).astype(np.uint8)

def affine(img, mask, r):
    M = cv2.getRotationMatrix2D((112, 112), r.uniform(-15, 15), r.uniform(.85, 1.1)); M[:, 2] += r.uniform(-15, 15, 2)
    return (cv2.warpAffine(img, M, (W, H), borderMode=cv2.BORDER_CONSTANT),
            cv2.warpAffine(mask, M, (W, H), flags=cv2.INTER_NEAREST))

def compose(img, mask, family, r, sleeve=True, aug=True):
    if aug: img, mask = affine(img, mask, r)
    bg = BG[family](r)
    if sleeve and r.random() < .5: bg = add_sleeve(bg, r)
    alpha = cv2.GaussianBlur(mask.astype(np.float32), (5, 5), 0)[..., None]
    hand = img.astype(np.float32)
    if aug: hand = np.clip(hand * r.uniform(.75, 1.2) + r.uniform(-15, 15), 0, 255)
    return (alpha * hand + (1 - alpha) * bg.astype(np.float32)).astype(np.uint8)

def watermark(img, text="SYNTHETIC BACKGROUND"):
    out = img.copy(); cv2.putText(out, text, (6, 215), cv2.FONT_HERSHEY_SIMPLEX, .5, (255, 0, 0), 2, cv2.LINE_AA); return out

In [ ]:
rng = np.random.default_rng(0)
synthX, synthy = [], []
for img, m, y in zip(Xtr, train_masks, ytr):
    if m is None: continue
    for _ in range(N_SYNTH_PER_IMG):
        synthX.append(compose(img, m, TRAIN_FAMS[rng.integers(len(TRAIN_FAMS))], rng)); synthy.append(y)
synthy = np.array(synthy)
share = len(synthX) / (len(synthX) + len(Xtr))
print(f"original train: {len(Xtr)} | synthetic: {len(synthX)} | synthetic share of training set: {share:.1%}")

fig, ax = plt.subplots(2, 8, figsize=(20, 5.2))
for j, i in enumerate(np.random.default_rng(2).choice(len(synthX), 16, replace=False)):
    a = ax[j // 8, j % 8]; a.imshow(synthX[i]); a.set_title(f"SYNTHETIC | {CLASSES[synthy[i]]}", fontsize=8, color="red"); a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "fig_synthetic_examples.png", dpi=120); plt.show()

## 4. Feature extractors: A1 backbone (MobileNetV2) vs SOTA (DINOv2)

In [ ]:
from transformers import AutoModel
dino = AutoModel.from_pretrained(DINO_NAME, attn_implementation="eager").to(dev).eval()
mn = torchvision.models.mobilenet_v2(weights=torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1).to(dev).eval()

tfm = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
to_batch = lambda imgs: torch.stack([tfm(Image.fromarray(i)) for i in imgs])

def dino_fn(x):
    h = dino(pixel_values=x).last_hidden_state
    return torch.cat([h[:, 0], h[:, 1:].mean(1)], 1)
def mnv2_fn(x):
    return mn.features(x).mean((2, 3))
BACKBONES = {"MobileNetV2": mnv2_fn, "DINOv2": dino_fn}

@torch.no_grad()
def feats(fn, imgs, bs=64):
    return torch.cat([fn(to_batch(imgs[i:i + bs]).to(dev)).float().cpu() for i in range(0, len(imgs), bs)]).numpy()

F = {b: {k: feats(fn, v) for k, v in [("train", Xtr), ("synth", synthX), ("val", Xva), ("test", Xte)]} for b, fn in BACKBONES.items()}
print({b: F[b]["train"].shape[1] for b in F})

## 5. Train linear heads (3 seeds) and compare
Both backbones are frozen, with the same linear head, optimizer and early stopping (on validation accuracy, patience 8), so the comparison is fair.

In [ ]:
class Head(nn.Module):
    def __init__(s, mu, sd, d, k=3):
        super().__init__(); s.register_buffer("mu", mu); s.register_buffer("sd", sd)
        s.drop = nn.Dropout(0.2); s.fc = nn.Linear(d, k)
    def forward(s, x): return s.fc(s.drop((x - s.mu) / s.sd))

def fit(Xa, ya, Xv, yv, seed, epochs=80, lr=1e-3, wd=1e-2, bs=64, patience=8):
    seed_all(seed)
    Xa, ya = torch.tensor(Xa).to(dev), torch.tensor(ya).long().to(dev); Xv, yv_t = torch.tensor(Xv).to(dev), torch.tensor(yv).long().to(dev)
    h = Head(Xa.mean(0), Xa.std(0) + 1e-6, Xa.shape[1]).to(dev)
    opt = torch.optim.AdamW(h.parameters(), lr=lr, weight_decay=wd); lossf = nn.CrossEntropyLoss()
    best, best_state, bad, hist = -1, None, 0, []
    for ep in range(epochs):
        h.train(); perm = torch.randperm(len(Xa), device=dev); tl = 0
        for i in range(0, len(Xa), bs):
            b = perm[i:i + bs]; opt.zero_grad(); l = lossf(h(Xa[b]), ya[b]); l.backward(); opt.step(); tl += l.item() * len(b)
        h.eval()
        with torch.no_grad(): va = (h(Xv).argmax(1) == yv_t).float().mean().item()
        hist.append((tl / len(Xa), va))
        if va > best: best, best_state, bad = va, copy.deepcopy(h.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break
    h.load_state_dict(best_state); return h.eval(), hist

@torch.no_grad()
def predict(h, X): return h(torch.tensor(X).to(dev)).argmax(1).cpu().numpy()

heads, hists, rows = {}, {}, []
for b in BACKBONES:
    for a in ["orig", "orig+synth"]:
        Xa = F[b]["train"] if a == "orig" else np.concatenate([F[b]["train"], F[b]["synth"]])
        ya = ytr if a == "orig" else np.concatenate([ytr, synthy])
        for s in SEEDS:
            h, hist = fit(Xa, ya, F[b]["val"], yva, s); heads[(b, a, s)] = h; hists[(b, a, s)] = hist
            p = predict(h, F[b]["test"]); f1s = f1_score(yte, p, average=None, labels=[0, 1, 2])
            rows.append(dict(backbone=b, train_data=a, seed=s, acc=accuracy_score(yte, p),
                             macro_f1=f1_score(yte, p, average="macro"), f1_peace=f1s[0], f1_stop=f1s[1], f1_thumbs_up=f1s[2]))
res = pd.DataFrame(rows); res.to_csv(OUT / "results_all_runs.csv", index=False)
summ = res.groupby(["backbone", "train_data"])[["acc", "macro_f1", "f1_peace", "f1_stop", "f1_thumbs_up"]].agg(["mean", "std"]).round(3)
summ.to_csv(OUT / "results_summary.csv"); summ

**For the report:** also quote the A1 numbers (frozen MobileNetV2 fine-tuned pipeline: acc 0.582 / 0.630, macro-F1 0.563 / 0.590 over two unseeded runs). The 2x2 table above is the ablation (with/without synthetic data) and the backbone comparison.

In [ ]:
# training curves (seed 0) + confusion matrices
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for (b, a, s), hist in hists.items():
    if s != 0: continue
    ax[0].plot([h[0] for h in hist], label=f"{b} {a}"); ax[1].plot([h[1] for h in hist], label=f"{b} {a}")
ax[0].set_title("Train loss"); ax[1].set_title("Validation accuracy"); ax[0].set_xlabel("epoch"); ax[1].set_xlabel("epoch"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "fig_training_curves.png", dpi=120); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a_, key in zip(ax, [("MobileNetV2", "orig", 0), ("DINOv2", "orig+synth", 0)]):
    cm = confusion_matrix(yte, predict(heads[key], F[key[0]]["test"]), labels=[0, 1, 2])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=a, cbar=False)
    a.set_title(f"{key[0]} / {key[1]}"); a.set_xlabel("predicted"); a.set_ylabel("true")
plt.tight_layout(); plt.savefig(OUT / "fig_confusion.png", dpi=120); plt.show()

## 6. Stress test: accuracy per background
The real test hands (cut by SAM2) are pasted on each background family. `photo` was **never** used in training.

In [ ]:
valid = [i for i, m in enumerate(test_masks) if m is not None]
r2 = np.random.default_rng(123)
stress_imgs = {fam: [compose(Xte[i], test_masks[i], fam, r2, sleeve=False, aug=False) for i in valid] for fam in BG}
y_valid = yte[valid]
stress = {}
for b, fn in BACKBONES.items():
    Fs = {fam: feats(fn, imgs) for fam, imgs in stress_imgs.items()}
    for a in ["orig", "orig+synth"]:
        accs = {"original test": np.mean([accuracy_score(yte, predict(heads[(b, a, s)], F[b]["test"])) for s in SEEDS])}
        for fam in BG:
            accs[fam + (" (unseen)" if fam == "photo" else "")] = np.mean([accuracy_score(y_valid, predict(heads[(b, a, s)], Fs[fam])) for s in SEEDS])
        stress[f"{b} / {a}"] = accs
stress_df = pd.DataFrame(stress).T.round(3); stress_df.to_csv(OUT / "stress_test.csv")
plt.figure(figsize=(10, 3.5)); sns.heatmap(stress_df, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1)
plt.title("Test accuracy per background (mean of 3 seeds)"); plt.tight_layout(); plt.savefig(OUT / "fig_stress_test.png", dpi=120); plt.show()
stress_df

## 7. Explainability: DINOv2 attention rollout

In [ ]:
@torch.no_grad()
def rollout(img):
    out = dino(pixel_values=to_batch([img]).to(dev), output_attentions=True)
    n = out.attentions[0].shape[-1]; r = torch.eye(n, device=dev)
    for a in out.attentions:
        a = a.mean(1)[0] + torch.eye(n, device=dev); a = a / a.sum(-1, keepdim=True); r = a @ r
    g = int((n - 1) ** 0.5); m = r[0, 1:].reshape(g, g).cpu().numpy(); m = (m - m.min()) / (m.max() - m.min() + 1e-8)
    return cv2.resize(m, (224, 224), interpolation=cv2.INTER_CUBIC)

def overlay(img, m):
    heat = cv2.applyColorMap((np.clip(m, 0, 1) * 255).astype(np.uint8), cv2.COLORMAP_JET)[..., ::-1]
    return (0.55 * img + 0.45 * heat).astype(np.uint8)

p = predict(heads[("DINOv2", "orig+synth", 0)], F["DINOv2"]["test"])
ok = np.where(p == yte)[0]; bad = np.where(p != yte)[0]
pick = list(np.random.default_rng(3).choice(ok, 4, replace=False)) + list(np.random.default_rng(4).choice(bad, min(4, len(bad)), replace=False))
fig, ax = plt.subplots(2, len(pick), figsize=(2.6 * len(pick), 5.5))
for j, i in enumerate(pick):
    ax[0, j].imshow(Xte[i]); ax[0, j].set_title(f"true {CLASSES[yte[i]]}\npred {CLASSES[p[i]]}", fontsize=9, color="green" if p[i] == yte[i] else "red")
    ax[1, j].imshow(overlay(Xte[i], rollout(Xte[i]))); ax[0, j].axis("off"); ax[1, j].axis("off")
plt.tight_layout(); plt.savefig(OUT / "fig_attention.png", dpi=120); plt.show()

*Note for the report:* the backbone is frozen, so the attention map shows what the DINOv2 features focus on (hand vs background); it does not change with the synthetic data. The effect of the synthetic data is shown by the metrics and the stress test.

## 8. Latency and save the final model

In [ ]:
final = heads[("DINOv2", "orig+synth", 0)].eval()
x = to_batch([Xte[0]]).to(dev)
def one():
    with torch.no_grad(): final(dino_fn(x))
for _ in range(10): one()
if dev == "cuda": torch.cuda.synchronize()
t0 = time.time()
for _ in range(50): one()
if dev == "cuda": torch.cuda.synchronize()
LAT = (time.time() - t0) / 50 * 1000
print(f"DINOv2 ({DINO_NAME}) + head: {LAT:.1f} ms / image on {DEV_NAME}")
torch.save({"head": final.state_dict(), "classes": CLASSES, "dino": DINO_NAME}, OUT / "head_dinov2_synth.pt")
json.dump({"device": DEV_NAME, "latency_ms": LAT, "synthetic_share": share, "n_train": len(Xtr), "n_synth": len(synthX)}, open(OUT / "meta.json", "w"))

## 9. Demo app (Gradio): upload / webcam -> prediction -> attention map -> latency

In [ ]:
import gradio as gr

def app_predict(image, swap, family):
    if image is None: return None, None, None, "Upload an image first."
    img = np.array(Image.fromarray(image).convert("RGB")); h, w = img.shape[:2]; s = min(h, w)
    img = cv2.resize(img[(h - s) // 2:(h - s) // 2 + s, (w - s) // 2:(w - s) // 2 + s], (224, 224))
    shown, note = img, ""
    if swap:
        m = hand_mask(img)
        if m is None: note = " | SAM2 could not isolate the hand, original image used."
        else:
            img = compose(img, m, family, np.random.default_rng(), sleeve=False, aug=False)
            shown = watermark(img); note = " | SYNTHETIC background (stress-test)"
    if dev == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad(): probs = final(dino_fn(to_batch([img]).to(dev))).softmax(1)[0].cpu().numpy()
    if dev == "cuda": torch.cuda.synchronize()
    ms = (time.time() - t0) * 1000
    return ({c: float(p) for c, p in zip(CLASSES, probs)}, shown, overlay(img, rollout(img)),
            f"Inference: {ms:.1f} ms on {DEV_NAME} (Colab, DINOv2 + linear head){note}")

demo = gr.Interface(
    fn=app_predict,
    inputs=[gr.Image(sources=["upload", "webcam"], label="Hand gesture photo"),
            gr.Checkbox(label="Stress-test: replace background (SAM2)"),
            gr.Dropdown(list(BG.keys()), value="curtain", label="Synthetic background")],
    outputs=[gr.Label(num_top_classes=3, label="Prediction"), gr.Image(label="Input used (watermarked if synthetic)"),
             gr.Image(label="Attention rollout (DINOv2)"), gr.Textbox(label="Info")],
    title="Hand gesture recognition: peace / stop / thumbs up",
    description="DINOv2 features + linear head, trained with SAM2 background-swap synthetic data. Synthetic images are watermarked. Do not upload photos of other people without their consent.")
demo.launch(share=True, debug=False)

## 10. Download all results (figures, tables, model) for the report and repo

In [ ]:
!cd /content && zip -qr outputs.zip outputs
files.download("/content/outputs.zip")